# YOLOv11 — Manga109 Text Detection Training Notebook

**Notebook huấn luyện mô hình YOLO11 nhận diện vùng chữ trong truyện tranh (Manga109)**

Môi trường mục tiêu: **UFO-Space / Cloud GPU** với card đồ họa **NVIDIA RTX 3080 (10GB VRAM)**

---

### Quy trình gồm 5 bước:
| Cell | Mục đích |
|------|----------|
| **Cell 1** | Cài đặt thư viện Ultralytics (YOLO11) |
| **Cell 2** | Xác nhận GPU — In tên card đồ họa (RTX 3080) |
| **Cell 3** | Khởi tạo mô hình YOLO11 Nano (`yolo11n.pt`) |
| **Cell 4** | Huấn luyện với tham số tối ưu cho RTX 3080 (`batch=32`, `amp=True`) |
| **Cell 5** | Inference — Dự đoán và hiển thị kết quả bằng Matplotlib |

> **Trước khi chạy notebook này**, hãy đảm bảo đã chạy `python scripts/prepare_data.py`
> để tạo thư mục `dataset/` với cấu trúc `images/train`, `images/val`, `labels/train`, `labels/val`.

---
## Cell 1 — Cài đặt thư viện Ultralytics (YOLO11)

In [ ]:
# Cài đặt thư viện Ultralytics chứa YOLO11
# -q : quiet mode (giảm log rác)
# --upgrade : đảm bảo luôn dùng phiên bản mới nhất
!pip install ultralytics -q --upgrade

# Xác nhận cài đặt thành công bằng cách in phiên bản
import ultralytics
print(f'[OK] Ultralytics version: {ultralytics.__version__}')

---
## Cell 2 — Xác nhận GPU (RTX 3080)

Nếu output hiển thị `NVIDIA GeForce RTX 3080` → sẵn sàng train.

In [ ]:
import torch

# In thông tin GPU qua nvidia-smi (lệnh shell của NVIDIA driver)
print('=' * 55)
print('  KIEM TRA GPU - nvidia-smi')
print('=' * 55)
!nvidia-smi

# Kiểm tra GPU thông qua PyTorch API
print('\n' + '=' * 55)
print('  KIEM TRA GPU - PyTorch CUDA')
print('=' * 55)
print(f'PyTorch Version  : {torch.__version__}')
print(f'CUDA Available   : {torch.cuda.is_available()}')

if torch.cuda.is_available():
    # torch.cuda.get_device_name(0) trả về tên đầy đủ của GPU ở slot 0
    # Kết quả mong đợi: 'NVIDIA GeForce RTX 3080'
    gpu_name   = torch.cuda.get_device_name(0)
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    vram_free  = torch.cuda.mem_get_info(0)[0] / (1024 ** 3)
    print(f'GPU Name         : {gpu_name}')
    print(f'Total VRAM       : {vram_total:.2f} GB')
    print(f'Free  VRAM       : {vram_free:.2f} GB')
    print(f'GPU Count        : {torch.cuda.device_count()}')
    print(f'cuDNN Version    : {torch.backends.cudnn.version()}')
    if '3080' in gpu_name:
        print('\n[OK] Da xac nhan GPU la NVIDIA RTX 3080! San sang train.')
    else:
        print(f'\n[*] GPU hien tai: {gpu_name}.')
else:
    print('[CANH BAO] Khong phat hien GPU! Train se chay tren CPU (rat cham).')
print('=' * 55)

---
## Cell 3 — Khởi tạo mô hình YOLO11 Nano (`yolo11n.pt`)

File `yolo11n.pt` đã có sẵn trong thư mục gốc của project (đã upload cùng khi nén .zip).

In [ ]:
import sys
from pathlib import Path
from ultralytics import YOLO

# Xác định thư mục gốc của project
# Notebook nằm trong notebooks/ → project_root là thư mục cha
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f'[*] Project root : {project_root}')

# Đường dẫn đến pretrained weights YOLO11 Nano
weights_path = project_root / 'yolo11n.pt'

if not weights_path.exists():
    # Nếu chưa có, Ultralytics tự tải từ internet
    print('[*] Khong tim thay yolo11n.pt. Se tu tai ve tu internet...')
    model = YOLO('yolo11n.pt')
else:
    print(f'[OK] Tim thay weights tai: {weights_path}')
    model = YOLO(str(weights_path))

print('\n[OK] Da khoi tao thanh cong YOLO11 Nano!')
print('\n--- Thong tin kien truc mo hinh ---')
model.info(verbose=False)

---
## Cell 4 — Huấn luyện mô hình (Train)

### Tham số tối ưu cho RTX 3080 (10 GB VRAM):

| Tham số | Giá trị | Lý do |
|---------|---------|-------|
| `epochs` | `50` | Đủ để hội tụ với Manga109 |
| `imgsz` | `640` | Kích thước chuẩn YOLO |
| `batch` | `32` | RTX 3080 10GB VRAM xử lý tốt |
| `workers` | `8` | 8 luồng CPU tải data song song |
| `amp` | `True` | FP16 Mixed Precision → nhanh ~2x, tiết kiệm VRAM |
| `device` | `0` | GPU slot 0 (RTX 3080) |
| `patience` | `15` | Early stopping, tránh overfitting |

In [ ]:
# Đường dẫn đến file YAML cấu hình dataset
yaml_path = project_root / 'manga_text.yaml'

if not yaml_path.exists():
    raise FileNotFoundError(
        f'Khong tim thay file cau hinh: {yaml_path.resolve()}\n'
        f'Hay dam bao manga_text.yaml nam o thu muc goc du an.'
    )

# Xác định thiết bị: GPU (0) nếu có, ngược lại CPU
train_device = 0 if torch.cuda.is_available() else 'cpu'

print('=' * 55)
print('  BAT DAU HUAN LUYEN YOLO11 - MANGA109 TEXT DETECTION')
print('=' * 55)
print(f'  Dataset config : {yaml_path}')
print(f'  Device         : {"GPU cuda:0 (RTX 3080)" if train_device == 0 else "CPU"}')
print(f'  Epochs / imgsz : 50 / 640')
print(f'  Batch / Workers: 32 / 8')
print(f'  AMP (FP16)     : True')
print('=' * 55 + '\n')

# Chạy huấn luyện với tham số tối ưu cho RTX 3080
results = model.train(
    data=str(yaml_path),           # File YAML định nghĩa dataset
    epochs=50,                      # Số vòng lặp huấn luyện
    imgsz=640,                      # Kích thước ảnh đầu vào (640x640)
    batch=32,                       # Batch size tối ưu cho RTX 3080 10GB
    workers=8,                      # Số luồng CPU tải ảnh song song
    amp=True,                       # Automatic Mixed Precision (FP16)
    device=train_device,            # 0 = GPU (RTX 3080)
    project=str(project_root / 'runs' / 'detect'),  # Thư mục lưu kết quả
    name='manga_text_yolo11n',      # Tên thư mục con của lần train này
    save=True,                      # Lưu checkpoint .pt
    plots=True,                     # Vẽ biểu đồ loss, metrics
    patience=15,                    # Early stopping
)

print('\n' + '=' * 55)
print('  [OK] HUAN LUYEN HOAN TAT!')
print('=' * 55)
print(f'  Ket qua  : runs/detect/manga_text_yolo11n/')
print(f'  Best .pt : runs/detect/manga_text_yolo11n/weights/best.pt')
print(f'  Last .pt : runs/detect/manga_text_yolo11n/weights/last.pt')
print('=' * 55)

---
## Cell 5 — Inference (Dự đoán & Hiển thị kết quả)

Nạp `best.pt` và chạy `model.predict()` trên ảnh mẫu từ tập Validation.  
Kết quả được vẽ bounding box và hiển thị ngay trong notebook.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# -----------------------------------------------------------------------
# BƯỚC 1: Nạp model tốt nhất từ quá trình huấn luyện
# -----------------------------------------------------------------------
best_weights = project_root / 'runs' / 'detect' / 'manga_text_yolo11n' / 'weights' / 'best.pt'

if best_weights.exists():
    # Nạp model với trọng số tốt nhất (best.pt - checkpoint có mAP cao nhất)
    inference_model = YOLO(str(best_weights))
    print(f'[OK] Da nap model tu: {best_weights}')
else:
    # Dùng model hiện tại nếu chưa train
    inference_model = model
    print('[*] Chua tim thay best.pt. Dung model hien tai.')

# -----------------------------------------------------------------------
# BƯỚC 2: Chọn ảnh nguồn để chạy inference
# -----------------------------------------------------------------------
val_dir = project_root / 'dataset' / 'images' / 'val'
val_imgs = sorted(list(val_dir.glob('*.jpg')) + list(val_dir.glob('*.png')))

if val_imgs:
    source_path = str(val_imgs[0])
    print(f'[*] Anh test: {val_imgs[0].name}')
else:
    # Fallback nếu chưa có dataset
    source_path = str(project_root / 'demo_samples')
    print(f'[!] Khong tim thay anh val. Thu muc demo_samples: {source_path}')

# -----------------------------------------------------------------------
# BƯỚC 3: Chạy dự đoán — model.predict()
# -----------------------------------------------------------------------
# conf=0.25 : Chỉ giữ bbox có confidence >= 25%
# save=True : Lưu ảnh đã vẽ bbox vào runs/detect/predict/
results = inference_model.predict(
    source=source_path,
    conf=0.25,          # Ngưỡng confidence tối thiểu để giữ bbox
    iou=0.45,           # Ngưỡng IoU cho Non-Maximum Suppression
    save=True,          # Lưu ảnh kết quả vào disk
    device=0 if torch.cuda.is_available() else 'cpu'
)

# -----------------------------------------------------------------------
# BƯỚC 4: Hiển thị kết quả bằng Matplotlib
# -----------------------------------------------------------------------
result    = results[0]            # Kết quả của ảnh đầu tiên
num_boxes = len(result.boxes)     # Số bbox được phát hiện

# result.plot() trả về numpy array BGR — cần chuyển sang RGB cho matplotlib
annotated_bgr = result.plot(line_width=2, labels=True)
annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 1, figsize=(14, 18))
ax.imshow(annotated_rgb)
ax.axis('off')
ax.set_title(
    f'YOLO11 Text Detection — Phat hien {num_boxes} vung chu',
    fontsize=16, fontweight='bold', pad=14
)
plt.tight_layout()
plt.show()

print(f'\n[OK] Inference hoan tat! Phat hien: {num_boxes} bounding box.')
print('     Anh output da luu vao: runs/detect/predict/')

# In chi tiết từng bbox
if num_boxes > 0:
    print(f'\n--- Chi tiet {num_boxes} bounding box ---')
    for i, box in enumerate(result.boxes):
        x1, y1, x2, y2 = [round(v) for v in box.xyxy[0].tolist()]
        conf = round(box.conf[0].item(), 3)
        print(f'  Box {i+1:>3}: class=0 (text), conf={conf:.3f}, '
              f'bbox=[{x1}, {y1}, {x2}, {y2}]')